# Grammar Scoring Engine: Final Report

## Overview
This project develops a Grammar Scoring Engine for spoken English audio samples, predicting a continuous grammar score (0-5) for each file. The solution combines deep audio embeddings (Wav2Vec2) and handcrafted speech features, using a robust ensemble regression pipeline.

---

## 1. Data & Problem Statement
- **Input:** 45-60s spoken English audio files (train/test split)
- **Output:** Continuous grammar score (0-5)
- **Evaluation:** Pearson Correlation (Leaderboard), RMSE (for internal validation)

---

## 2. Preprocessing
- **Silence Removal:** `librosa.effects.trim` removes leading/trailing silence.
- **Noise Reduction:** Preemphasis filtering reduces background noise.
- **Amplitude Normalization:** Each audio is normalized to a fixed RMS value.
- All steps are applied identically to train and test audio.

---

## 3. Feature Extraction
- **Wav2Vec2 Embeddings:**
  - HuggingFace's `facebook/wav2vec2-base-960h` model.
  - Mean-pooled hidden states (768-dim) per audio file.
- **Handcrafted Features:**
  - MFCCs (mean, std), delta MFCCs, spectral contrast, chroma, pitch (mean, std), ZCR, RMS, speech rate, pause fraction.
- **Feature Fusion:** All features are concatenated into a single vector per file.

---

## 4. Feature Selection & Dimensionality Reduction
- **Constant Feature Removal:** `VarianceThreshold` drops zero-variance features.
- **PCA:** 128 principal components (fit on train fold only).
- **StandardScaler:** Standardizes features after PCA.

---

## 5. Model Pipeline & Ensembling
- **5-Fold Cross-Validation:** For robust validation and blending.
- **Models:**
  - LightGBM Regressor (regularized)
  - Ridge Regression
  - Support Vector Regression (SVR)
- **Ensembling:** Simple average of all three models' predictions.

---

## 6. Evaluation Results
- **Training RMSE (OOF, 5-fold blend):** ~0.82
- **Public Leaderboard Score:** ~0.56 (Pearson correlation)

> **Note:** Reporting RMSE on the training data is compulsory for submission.

- The model generalizes well on validation, but the public score suggests a domain gap or label noise in the test set.
- Wav2Vec2 embeddings fused with rich handcrafted features provided the best results among all tested approaches.

---

## 7. Visualizations

### Training History
![Training History](training_history.png)

### Feature Importance
![Feature Importance](feature_importance.png)

---

## 8. Key Takeaways & Future Work
- **Audio-only grammar scoring is challenging** without transcripts; linguistic features from ASR could further boost performance.
- **Further improvements:**
  - Use more advanced speech embeddings (e.g., WavLM, HuBERT)
  - Incorporate ASR transcripts and grammar-checker features
  - Hyperparameter tuning and stacking/weighted ensembling
  - Data augmentation to match test distribution

---

## 9. Evaluation Criteria
- **Correctness:** Does the solution work as expected?
- **Code Quality:** Is the code clean, well-structured, and documented?
- **Performance:** How well does the model perform on the test dataset?
- **Interpretability:** Are the results well-explained with relevant visualizations?

---

## 10. Reproducibility
- All code is in `main.py`.
- Required packages: `librosa`, `parselmouth`, `lightgbm`, `scikit-learn`, `transformers`, `torch`, `torchaudio`.
- To run: `python main.py`

---

In [7]:
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras import layers, models
import os
import warnings
from sklearn.mixture import GaussianMixture
import tensorflow_hub as hub
import lightgbm as lgb
import parselmouth
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.model_selection import KFold
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import torch
import torchaudio
warnings.filterwarnings('ignore')

In [8]:
YAMNET_MODEL_HANDLE = 'https://tfhub.dev/google/yamnet/1'
yamnet_model = hub.load(YAMNET_MODEL_HANDLE)

# Load Wav2Vec2 model and processor globally
W2V2_MODEL_NAME = 'facebook/wav2vec2-base-960h'
w2v2_processor = Wav2Vec2Processor.from_pretrained(W2V2_MODEL_NAME)
w2v2_model = Wav2Vec2Model.from_pretrained(W2V2_MODEL_NAME)
w2v2_model.eval()


Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wav2Vec2Model(
  (feature_extractor): Wav2Vec2FeatureEncoder(
    (conv_layers): ModuleList(
      (0): Wav2Vec2GroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): Wav2Vec2FeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Wav2Vec2Encoder(
    (pos_conv_embed): Wav2Vec2PositionalConvEmbedding(
  

In [9]:
def preprocess_audio(y, sr):
    # Remove leading/trailing silence
    y, _ = librosa.effects.trim(y, top_db=20)
    # Preemphasis for noise reduction
    y = librosa.effects.preemphasis(y)
    # Normalize amplitude to fixed RMS
    rms = np.sqrt(np.mean(y**2))
    if rms > 0:
        y = y / rms
    return y

In [10]:
def extract_yamnet_embedding(audio_path):
    import soundfile as sf
    # Load audio and resample to 16kHz mono
    wav, sr = sf.read(audio_path)
    if len(wav.shape) > 1:
        wav = wav.mean(axis=1)  # Convert to mono
    if sr != 16000:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=16000)
        sr = 16000
    # Preprocess
    wav = preprocess_audio(wav, sr)
    # Convert to float32
    wav = wav.astype('float32')
    # Run YAMNet
    scores, embeddings, spectrogram = yamnet_model(wav)
    emb = embeddings.numpy()
    emb_mean = emb.mean(axis=0)
    return emb_mean

In [11]:
def extract_handcrafted_features(audio_path):
    import soundfile as sf
    y, sr = sf.read(audio_path)
    if len(y.shape) > 1:
        y = y.mean(axis=1)
    y = preprocess_audio(y, sr)
    # MFCCs
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfccs, axis=1)
    mfcc_std = np.std(mfccs, axis=1)
    # Delta MFCCs
    mfcc_delta = librosa.feature.delta(mfccs)
    mfcc_delta_mean = np.mean(mfcc_delta, axis=1)
    mfcc_delta_std = np.std(mfcc_delta, axis=1)
    # Spectral contrast
    spec_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    spec_contrast_mean = np.mean(spec_contrast, axis=1)
    spec_contrast_std = np.std(spec_contrast, axis=1)
    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)
    chroma_std = np.std(chroma, axis=1)
    # ZCR
    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_mean = np.mean(zcr)
    # RMS
    rms = librosa.feature.rms(y=y)
    rms_mean = np.mean(rms)
    # Pitch (using parselmouth)
    snd = parselmouth.Sound(y, sr)
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]
    pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    pitch_std = np.std(pitch_values) if len(pitch_values) > 0 else 0
    # Speech rate (roughly: #zero-crossings per second)
    speech_rate = zcr_mean * sr
    # Pause stats (fraction of near-silence frames)
    silence_mask = np.abs(y) < 0.02
    pause_fraction = np.mean(silence_mask)
    # Feature vector
    features = np.concatenate([
        mfcc_mean, mfcc_std, mfcc_delta_mean, mfcc_delta_std,
        spec_contrast_mean, spec_contrast_std,
        chroma_mean, chroma_std,
        [zcr_mean, rms_mean, pitch_mean, pitch_std, speech_rate, pause_fraction]
    ])
    return features

In [12]:
def extract_wav2vec2_embedding(audio_path):
    waveform, sr = torchaudio.load(audio_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)  # Convert to mono
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(waveform)
        sr = 16000
    waveform = waveform.squeeze().numpy()
    # Preprocess (use your existing preprocess_audio)
    waveform = preprocess_audio(waveform, sr)
    # Convert to torch tensor
    input_values = w2v2_processor(waveform, sampling_rate=16000, return_tensors="pt").input_values
    with torch.no_grad():
        hidden_states = w2v2_model(input_values).last_hidden_state
    emb = hidden_states.mean(dim=1).squeeze().numpy()  # mean-pool over time
    return emb

In [13]:
def process_audio_files_fusion(df, audio_dir):
    features_list = []
    for audio_file in df['filename']:
        audio_path = os.path.join(audio_dir, audio_file)
        try:
            yamnet_emb = extract_yamnet_embedding(audio_path)
            hand_feats = extract_handcrafted_features(audio_path)
            features = np.concatenate([yamnet_emb, hand_feats])
            features_list.append(features)
        except Exception as e:
            print(f"Error processing {audio_file}: {str(e)}")
            features_list.append(np.zeros(1024+13+13+4))  # fallback
    features_df = pd.DataFrame(features_list)
    return features_df

In [14]:
def process_audio_files_fusion_w2v2(df, audio_dir):
    features_list = []
    for audio_file in df['filename']:
        audio_path = os.path.join(audio_dir, audio_file)
        try:
            w2v2_emb = extract_wav2vec2_embedding(audio_path)
            hand_feats = extract_handcrafted_features(audio_path)
            features = np.concatenate([w2v2_emb, hand_feats])
            features_list.append(features)
        except Exception as e:
            print(f"Error processing {audio_file}: {str(e)}")
            features_list.append(np.zeros(768 + len(hand_feats)))  # fallback
    features_df = pd.DataFrame(features_list)
    return features_df

In [15]:
def main():
    # Load data
    print("Loading data...")
    train_df = pd.read_csv('shl-intern-hiring-assessment\\Dataset\\train.csv')
    test_df = pd.read_csv('shl-intern-hiring-assessment\\Dataset\\test.csv')
    print(f"Training data shape: {train_df.shape}")
    print(f"Test data shape: {test_df.shape}")
    # Process audio files with feature fusion
    print("Extracting fused features for training audio files...")
    train_audio_dir = 'shl-intern-hiring-assessment\\Dataset\\audios\\train'
    train_features = process_audio_files_fusion(train_df, train_audio_dir)
    train_features['score'] = train_df['label']
    X_full = train_features.drop('score', axis=1).values
    y_full = train_features['score'].values
    # Remove constant features
    print("Removing constant features...")
    selector = VarianceThreshold(threshold=0.0)
    X_full = selector.fit_transform(X_full)
    # Prepare test features
    print("Extracting fused features for test audio files...")
    test_audio_dir = 'shl-intern-hiring-assessment\\Dataset\\audios\\test'
    test_features = process_audio_files_fusion(test_df, test_audio_dir)
    X_test = test_features.values
    X_test = selector.transform(X_test)
    # K-Fold Cross-Validation
    print("Running 5-fold cross-validation and ensembling...")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds_lgbm = np.zeros(len(y_full))
    oof_preds_ridge = np.zeros(len(y_full))
    oof_preds_svr = np.zeros(len(y_full))
    test_preds_lgbm = np.zeros(len(X_test))
    test_preds_ridge = np.zeros(len(X_test))
    test_preds_svr = np.zeros(len(X_test))
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_full)):
        print(f"Fold {fold+1}/5")
        X_train, X_val = X_full[train_idx], X_full[val_idx]
        y_train, y_val = y_full[train_idx], y_full[val_idx]
        # PCA (fit only on train)
        pca = PCA(n_components=128, random_state=42)
        X_train_pca = pca.fit_transform(X_train)
        X_val_pca = pca.transform(X_val)
        X_test_pca = pca.transform(X_test)
        # StandardScaler (fit only on train)
        scaler = StandardScaler()
        X_train_pca = scaler.fit_transform(X_train_pca)
        X_val_pca = scaler.transform(X_val_pca)
        X_test_pca_scaled = scaler.transform(X_test_pca)
        # LightGBM
        lgbm = lgb.LGBMRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=5,
            num_leaves=16,
            min_child_samples=30,
            feature_fraction=0.8,
            bagging_fraction=0.8,
            lambda_l1=1.0,
            lambda_l2=1.0,
            random_state=42
        )
        lgbm.fit(
            X_train_pca, y_train,
            eval_set=[(X_val_pca, y_val)],
            callbacks=[lgb.early_stopping(20)]
        )
        oof_preds_lgbm[val_idx] = lgbm.predict(X_val_pca)
        test_preds_lgbm += lgbm.predict(X_test_pca_scaled) / kf.n_splits
        # Ridge
        ridge = Ridge(alpha=1.0)
        ridge.fit(X_train_pca, y_train)
        oof_preds_ridge[val_idx] = ridge.predict(X_val_pca)
        test_preds_ridge += ridge.predict(X_test_pca_scaled) / kf.n_splits
        # SVR
        svr = SVR(C=1.0, epsilon=0.2)
        svr.fit(X_train_pca, y_train)
        oof_preds_svr[val_idx] = svr.predict(X_val_pca)
        test_preds_svr += svr.predict(X_test_pca_scaled) / kf.n_splits
    # Blend predictions (simple average)
    oof_blend = (oof_preds_lgbm + oof_preds_ridge + oof_preds_svr) / 3
    test_blend = (test_preds_lgbm + test_preds_ridge + test_preds_svr) / 3
    # Evaluate
    rmse = np.sqrt(mean_squared_error(y_full, oof_blend))
    print(f"OOF RMSE (5-fold blend): {rmse:.4f}")
    # Create submission file
    submission = pd.DataFrame({
        'filename': test_df['filename'],
        'label': test_blend.flatten()
    })
    submission.to_csv('submission.csv', index=False)
    print("Submission file created successfully!")

if __name__ == "__main__":
    main() 

Loading data...
Training data shape: (444, 2)
Test data shape: (204, 1)
Extracting fused features for training audio files...
Removing constant features...
Extracting fused features for test audio files...
Running 5-fold cross-validation and ensembling...
Fold 1/5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 wi